In [ ]:
from pathlib import Path

LAST_NAME = "DAMOLUAN"
STUDENT_ID = "TUPM-26-3518"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)

In [ ]:
# EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS

import csv
import json
import shutil
import re
from pathlib import Path
from collections import Counter

# Folders
# The notebook and weblog.csv are in the same folder
ADDITIONAL_DIR = Path(".")
OUTPUT_DIR = ADDITIONAL_DIR / "output"
BACKUP_DIR = ADDITIONAL_DIR / "backup"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Student-specific settings
records_limit = 1000 + ID_SUM
target_class = (SEED_DIGIT % 4) + 2

# Dataset path
dataset_path = Path("weblog.csv")

# Output paths
filtered_path = OUTPUT_DIR / "web_log_filtered.csv"
summary_path = OUTPUT_DIR / "web_log_summary.json"
backup_path = BACKUP_DIR / "web_log_summary.json"

# Check if dataset exists
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {dataset_path.resolve()}"
    )

# Function to find dataset columns
def find_column(headers, choices):
    normalized = {
        re.sub(r"[^a-z0-9]", "", h.lower()): h
        for h in headers
    }

    for choice in choices:
        key = re.sub(r"[^a-z0-9]", "", choice.lower())

        if key in normalized:
            return normalized[key]

    return None


# Initialize variables
records_processed = 0
invalid_records = 0
selected_records = []
status_counts = Counter()
url_counts = Counter()

# Read dataset
with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:

    reader = csv.DictReader(file)
    headers = reader.fieldnames

    if not headers:
        raise ValueError("The dataset is empty or has no headers.")

    # Identify required columns
    ip_col = find_column(headers, ["IP", "IP Address"])
    time_col = find_column(headers, ["Time", "Timestamp"])
    url_col = find_column(headers, ["URL", "Requested URL"])
    status_col = find_column(
    headers, ["Status", "Staus", "Status Code", "HTTP Status"]
)

    if not all([ip_col, time_col, url_col, status_col]):
        raise ValueError(
            "Required columns are missing from weblog.csv. "
            f"Available columns: {headers}"
        )

    # Process records
    for row in reader:

        if records_processed >= records_limit:
            break

        records_processed += 1

        try:
            ip = row.get(ip_col, "").strip()
            timestamp = row.get(time_col, "").strip()
            url = row.get(url_col, "").strip()
            status = row.get(status_col, "").strip()

            # Check missing values
            if not all([ip, timestamp, url, status]):
                invalid_records += 1
                continue

            # Convert status to integer
            status_code = int(status)

            # Validate HTTP status code
            if not 100 <= status_code <= 599:
                invalid_records += 1
                continue

            # Select assigned HTTP status class
            if status_code // 100 == target_class:

                selected_records.append(row)
                status_counts[str(status_code)] += 1
                url_counts[url] += 1

        except (ValueError, TypeError):
            invalid_records += 1


# Export filtered records to CSV
with open(filtered_path, "w", encoding="utf-8", newline="") as file:

    writer = csv.DictWriter(file, fieldnames=headers)
    writer.writeheader()
    writer.writerows(selected_records)


# Identify five most frequently requested URLs
top_urls = [
    {"url": url, "requests": count}
    for url, count in url_counts.most_common(5)
]


# Create JSON summary
summary = {
    "student_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "records_processed": records_processed,
    "target_status_class": f"{target_class}xx",
    "records_in_target_class": len(selected_records),
    "invalid_records": invalid_records,
    "count_per_status_code": dict(status_counts),
    "five_most_requested_urls": top_urls,
    "filtered_csv_file": str(filtered_path)
}


# Save JSON summary
with open(summary_path, "w", encoding="utf-8") as file:

    json.dump(summary, file, indent=4)


# Create backup of JSON summary
shutil.copy2(summary_path, backup_path)

# Verify backup
backup_verified = (
    backup_path.exists()
    and backup_path.stat().st_size == summary_path.stat().st_size
)


# Display results
print("WEB SERVER ACCESS LOG ANALYSIS")
print("=" * 40)

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)

print("\nDataset Information:")
print("Dataset:", dataset_path.resolve())
print("Records processed:", records_processed)
print("Target status class:", f"{target_class}xx")
print("Records in target class:", len(selected_records))
print("Invalid records:", invalid_records)

print("\nCount per status code:")

for status, count in sorted(status_counts.items()):
    print(status, ":", count)

print("\nFive most requested URLs:")

for item in top_urls:
    print(item["url"], ":", item["requests"])

print("\nOUTPUT FILES")
print("-" * 40)
print("Filtered CSV:", filtered_path.resolve())
print("JSON summary:", summary_path.resolve())
print("Backup:", backup_path.resolve())
print("Backup verified:", backup_verified)



WEB SERVER ACCESS LOG ANALYSIS
Student: DAMOLUAN
Student ID: TUPM-26-3518
Seed Digit: 8
ID Sum: 25

Dataset Information:
Dataset: C:\Users\nikeg\Lab6_File_Handling_Additional_Exercises\weblog.csv
Records processed: 1025
Target status class: 2xx
Records in target class: 855
Invalid records: 0

Count per status code:
200 : 847
206 : 8

Five most requested URLs:
GET /login.php HTTP/1.1 : 105
GET /home.php HTTP/1.1 : 91
GET /css/normalize.css HTTP/1.1 : 39
GET /css/bootstrap.min.css HTTP/1.1 : 38
GET /css/font-awesome.min.css HTTP/1.1 : 37

OUTPUT FILES
----------------------------------------
Filtered CSV: C:\Users\nikeg\Lab6_File_Handling_Additional_Exercises\output\web_log_filtered.csv
JSON summary: C:\Users\nikeg\Lab6_File_Handling_Additional_Exercises\output\web_log_summary.json
Backup: C:\Users\nikeg\Lab6_File_Handling_Additional_Exercises\backup\web_log_summary.json
Backup verified: True


In [19]:
# EXERCISE 2: MOBILE DEVICE USAGE PROFILING

import csv
import json
import shutil
import re
from pathlib import Path

# Folders
ADDITIONAL_DIR = Path(".")
OUTPUT_DIR = ADDITIONAL_DIR / "output"
BACKUP_DIR = ADDITIONAL_DIR / "backup"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Student-specific criteria
target_os = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
target_behavior = (SEED_DIGIT % 5) + 1

# Dataset path
dataset_path = Path("user_behavior_dataset.csv")

# Output paths
filtered_path = OUTPUT_DIR / "mobile_usage_filtered.csv"
summary_path = OUTPUT_DIR / "mobile_usage_summary.json"
backup_path = BACKUP_DIR / "mobile_usage_filtered.csv"

# Check dataset
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {dataset_path.resolve()}"
    )

# Function to identify columns
def find_column(headers, choices):

    normalized = {
        re.sub(r"[^a-z0-9]", "", h.lower()): h
        for h in headers
    }

    for choice in choices:

        key = re.sub(r"[^a-z0-9]", "", choice.lower())

        if key in normalized:
            return normalized[key]

    return None


# Initialize variables
total_records = 0
os_matching_records = 0
selected_records = []

battery_values = []
data_values = []

invalid_battery = 0
invalid_data = 0

# Required exported fields
export_fields = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)"
]

# Read dataset
with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:

    reader = csv.DictReader(file)
    headers = reader.fieldnames

    if not headers:
        raise ValueError("Dataset is empty or has no headers.")

    # Identify required columns
    user_col = find_column(headers, ["User ID"])
    device_col = find_column(headers, ["Device Model"])
    os_col = find_column(headers, ["Operating System"])

    behavior_col = find_column(
        headers, ["User Behavior Class"]
    )

    battery_col = find_column(
        headers, ["Battery Drain (mAh/day)", "Battery Drain"]
    )

    data_col = find_column(
        headers, ["Data Usage (MB/day)", "Data Usage"]
    )

    if not all([
        user_col,
        device_col,
        os_col,
        behavior_col,
        battery_col,
        data_col
    ]):
        raise ValueError(
            f"Required columns are missing. Available columns: {headers}"
        )

    # Process records
    for row in reader:

        total_records += 1

        operating_system = row.get(os_col, "").strip()

        # Filter by operating system
        if operating_system.lower() != target_os.lower():
            continue

        os_matching_records += 1

        # Check behavior class
        try:
            behavior_class = int(
                row.get(behavior_col, "").strip()
            )
        except (ValueError, TypeError):
            continue

        if behavior_class == target_behavior:

            # Store selected user fields
            selected_records.append({
                "User ID": row.get(user_col, ""),
                "Device Model": row.get(device_col, ""),
                "Operating System": operating_system,
                "Battery Drain (mAh/day)": row.get(battery_col, ""),
                "Data Usage (MB/day)": row.get(data_col, "")
            })

            # Calculate battery drain
            try:
                battery = float(row[battery_col])

                if battery >= 0:
                    battery_values.append(battery)
                else:
                    invalid_battery += 1

            except (ValueError, TypeError):
                invalid_battery += 1

            # Calculate mobile data usage
            try:
                data_usage = float(row[data_col])

                if data_usage >= 0:
                    data_values.append(data_usage)
                else:
                    invalid_data += 1

            except (ValueError, TypeError):
                invalid_data += 1


# Calculate averages
average_battery = (
    sum(battery_values) / len(battery_values)
    if battery_values else 0
)

average_data = (
    sum(data_values) / len(data_values)
    if data_values else 0
)


# Export selected records to CSV
with open(filtered_path, "w", encoding="utf-8", newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=export_fields
    )

    writer.writeheader()
    writer.writerows(selected_records)


# Verify exported row count
with open(filtered_path, "r", encoding="utf-8", newline="") as file:

    exported_count = sum(
        1 for row in csv.DictReader(file)
    )

count_verified = exported_count == len(selected_records)


# Create JSON summary
summary = {
    "student_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "operating_system": target_os,
    "target_behavior_class": target_behavior,
    "total_records_in_dataset": total_records,
    "records_matching_operating_system": os_matching_records,
    "records_in_target_behavior_class": len(selected_records),
    "average_battery_drain_mAh_per_day": round(average_battery, 2),
    "average_data_usage_MB_per_day": round(average_data, 2),
    "valid_battery_measurements": len(battery_values),
    "valid_data_measurements": len(data_values),
    "invalid_battery_measurements": invalid_battery,
    "invalid_data_measurements": invalid_data,
    "exported_row_count": exported_count,
    "count_verification": "VERIFIED" if count_verified else "CHECK",
    "exported_csv_file": str(filtered_path)
}


# Save JSON summary
with open(summary_path, "w", encoding="utf-8") as file:

    json.dump(summary, file, indent=4)


# Create backup of exported CSV
shutil.copy2(filtered_path, backup_path)

# Verify backup and file size
backup_verified = (
    backup_path.exists()
    and backup_path.stat().st_size == filtered_path.stat().st_size
)

backup_size = (
    backup_path.stat().st_size
    if backup_verified else 0
)


# Display results
print("MOBILE DEVICE USAGE PROFILING")
print("=" * 40)

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)

print("\nFiltering Criteria:")
print("Operating System:", target_os)
print("Target Behavior Class:", target_behavior)

print("\nDataset Results:")
print("Total records:", total_records)
print("Records matching OS:", os_matching_records)
print("Records in target behavior class:", len(selected_records))

print("\nUsage Statistics:")
print("Average Battery Drain:", round(average_battery, 2), "mAh/day")
print("Average Data Usage:", round(average_data, 2), "MB/day")

print("\nVerification:")
print("Exported row count:", exported_count)
print("Count verification:", "VERIFIED" if count_verified else "CHECK")
print("Invalid battery measurements:", invalid_battery)
print("Invalid data measurements:", invalid_data)

print("\nOutput Files:")
print("Filtered CSV:", filtered_path.resolve())
print("JSON Summary:", summary_path.resolve())
print("Backup:", backup_path.resolve())
print("Backup verified:", backup_verified)
print("Backup file size:", backup_size, "bytes")

MOBILE DEVICE USAGE PROFILING
Student: DAMOLUAN
Student ID: TUPM-26-3518
Seed Digit: 8

Filtering Criteria:
Operating System: Android
Target Behavior Class: 4

Dataset Results:
Total records: 700
Records matching OS: 554
Records in target behavior class: 110

Usage Statistics:
Average Battery Drain: 2101.22 mAh/day
Average Data Usage: 1235.07 MB/day

Verification:
Exported row count: 110
Count verification: VERIFIED
Invalid battery measurements: 0
Invalid data measurements: 0

Output Files:
Filtered CSV: C:\Users\nikeg\Lab6_File_Handling_Additional_Exercises\output\mobile_usage_filtered.csv
JSON Summary: C:\Users\nikeg\Lab6_File_Handling_Additional_Exercises\output\mobile_usage_summary.json
Backup: C:\Users\nikeg\Lab6_File_Handling_Additional_Exercises\backup\mobile_usage_filtered.csv
Backup verified: True
Backup file size: 4188 bytes


In [20]:

# EXERCISE 3: CYBERSECURITY INCIDENT REPORT

import csv
import json
import shutil
from pathlib import Path
from collections import Counter

# Folders
ADDITIONAL_DIR = Path(".")
OUTPUT_DIR = ADDITIONAL_DIR / "output"
BACKUP_DIR = ADDITIONAL_DIR / "backup"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Student-specific settings
target_year = 2015 + SEED_DIGIT

# Dataset path
dataset_path = Path("Global_Cybersecurity_Threats_2015-2024.csv")

# Output paths
filtered_path = OUTPUT_DIR / "cybersecurity_filtered.csv"
summary_path = OUTPUT_DIR / "cybersecurity_summary.json"
backup_path = BACKUP_DIR / "cybersecurity_summary.json"
report_path = OUTPUT_DIR / "cybersecurity_report.txt"

# Check dataset
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {dataset_path.resolve()}"
    )


# Initialize variables
total_records = 0
target_records = []
attack_counts = Counter()
financial_losses = []
invalid_years = 0
invalid_losses = 0


# Read cybersecurity dataset
with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:

    reader = csv.DictReader(file)
    headers = reader.fieldnames

    if not headers:
        raise ValueError("Dataset is empty or has no headers.")

    # Identify required columns
    year_col = next(
        (h for h in headers if h.strip().lower() == "year"),
        None
    )

    attack_col = next(
        (h for h in headers if h.strip().lower() == "attack type"),
        None
    )

    loss_col = next(
        (
            h for h in headers
            if h.strip().lower() in [
                "financial loss (in million $)",
                "financial loss",
                "financial loss (million $)"
            ]
        ),
        None
    )

    if not all([year_col, attack_col, loss_col]):
        raise ValueError(
            f"Required columns are missing. Available columns: {headers}"
        )

    # Process records
    for row in reader:

        total_records += 1

        # Validate year
        try:
            year = int(row.get(year_col, "").strip())
        except (ValueError, TypeError):
            invalid_years += 1
            continue

        # Select incidents in target year
        if year == target_year:

            target_records.append(row)

            # Count attack types
            attack_type = row.get(attack_col, "").strip()

            if attack_type:
                attack_counts[attack_type] += 1

            # Validate financial loss
            try:
                loss = float(
                    row.get(loss_col, "").replace(",", "")
                    .replace("$", "").strip()
                )

                if loss >= 0:
                    financial_losses.append({
                        "record": row,
                        "loss": loss
                    })
                else:
                    invalid_losses += 1

            except (ValueError, TypeError):
                invalid_losses += 1


# Calculate financial loss statistics
total_loss = sum(
    item["loss"] for item in financial_losses
)

average_loss = (
    total_loss / len(financial_losses)
    if financial_losses else 0
)


# Identify five incidents with highest financial losses
# Python's stable sorting preserves original dataset order
# when two incidents have equal financial losses.
top_five = sorted(
    financial_losses,
    key=lambda item: item["loss"],
    reverse=True
)[:5]


# Export selected incidents to CSV
with open(filtered_path, "w", encoding="utf-8", newline="") as file:

    writer = csv.DictWriter(file, fieldnames=headers)
    writer.writeheader()
    writer.writerows(target_records)


# Prepare top five incidents for JSON
top_incidents = []

for item in top_five:

    incident = dict(item["record"])
    incident["financial_loss_numeric"] = item["loss"]

    top_incidents.append(incident)


# Create JSON summary
summary = {
    "student_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "target_year": target_year,
    "total_records_in_dataset": total_records,
    "incidents_in_target_year": len(target_records),
    "count_per_attack_type": dict(attack_counts),
    "total_financial_loss_million_usd": round(total_loss, 2),
    "average_financial_loss_million_usd": round(average_loss, 2),
    "valid_financial_loss_records": len(financial_losses),
    "invalid_year_records": invalid_years,
    "invalid_financial_loss_records": invalid_losses,
    "five_costliest_incidents": top_incidents,
    "filtered_csv_file": str(filtered_path)
}


# Save JSON summary
with open(summary_path, "w", encoding="utf-8") as file:

    json.dump(summary, file, indent=4)


# Create backup of JSON summary
shutil.copy2(summary_path, backup_path)

# Verify backup
backup_verified = (
    backup_path.exists()
    and backup_path.stat().st_size == summary_path.stat().st_size
)


# Generate text report
report_lines = []

report_lines.append("CYBERSECURITY INCIDENT REPORT")
report_lines.append("=" * 45)
report_lines.append(f"Student: {LAST_NAME}")
report_lines.append(f"Student ID: {STUDENT_ID}")
report_lines.append(f"Target Year: {target_year}")
report_lines.append("")

report_lines.append("DATASET INFORMATION")
report_lines.append("-" * 45)
report_lines.append(f"Total records: {total_records}")
report_lines.append(f"Incidents in target year: {len(target_records)}")
report_lines.append(f"Invalid year records: {invalid_years}")
report_lines.append("")

report_lines.append("INCIDENT COUNT PER ATTACK TYPE")
report_lines.append("-" * 45)

for attack, count in sorted(attack_counts.items()):
    report_lines.append(f"{attack}: {count}")

report_lines.append("")
report_lines.append("FINANCIAL LOSS ANALYSIS")
report_lines.append("-" * 45)
report_lines.append(
    f"Total financial loss: ${total_loss:.2f} million"
)
report_lines.append(
    f"Average financial loss: ${average_loss:.2f} million"
)
report_lines.append(
    f"Valid financial loss records: {len(financial_losses)}"
)
report_lines.append(
    f"Invalid financial loss records: {invalid_losses}"
)

report_lines.append("")
report_lines.append("FIVE COSTLIEST INCIDENTS")
report_lines.append("-" * 45)

for index, item in enumerate(top_five, start=1):

    attack = item["record"].get(attack_col, "")
    loss = item["loss"]

    report_lines.append(
        f"{index}. {attack} - ${loss:.2f} million"
    )

report_lines.append("")
report_lines.append("GENERATED FILES")
report_lines.append("-" * 45)
report_lines.append(f"Filtered CSV: {filtered_path}")
report_lines.append(f"JSON Summary: {summary_path}")
report_lines.append(f"JSON Backup: {backup_path}")
report_lines.append(f"Backup verified: {backup_verified}")


# Save text report
with open(report_path, "w", encoding="utf-8") as file:

    file.write("\n".join(report_lines))


# Display results
print("\n".join(report_lines))
print("\nReport file:", report_path.resolve())

CYBERSECURITY INCIDENT REPORT
Student: DAMOLUAN
Student ID: TUPM-26-3518
Target Year: 2023

DATASET INFORMATION
---------------------------------------------
Total records: 3000
Incidents in target year: 315
Invalid year records: 0

INCIDENT COUNT PER ATTACK TYPE
---------------------------------------------
DDoS: 51
Malware: 55
Man-in-the-Middle: 50
Phishing: 46
Ransomware: 50
SQL Injection: 63

FINANCIAL LOSS ANALYSIS
---------------------------------------------
Total financial loss: $15958.08 million
Average financial loss: $50.66 million
Valid financial loss records: 315
Invalid financial loss records: 0

FIVE COSTLIEST INCIDENTS
---------------------------------------------
1. Malware - $99.72 million
2. Man-in-the-Middle - $99.71 million
3. DDoS - $99.64 million
4. DDoS - $99.62 million
5. SQL Injection - $99.19 million

GENERATED FILES
---------------------------------------------
Filtered CSV: output\cybersecurity_filtered.csv
JSON Summary: output\cybersecurity_summary.json
JS